Scratch: working distance (exploration, not a decision)

The fine-details exploration suggested that a piece is recognized when the support set has an example of it at the scale of use. This notebook tests that rule with annotations that already exist: every example of a piece, in the candidates of the support-set study on V2, at least as large as the typical size of that piece in use is added to the current support set.

The typical size is the median number of patches of the piece over the 32 validation frames where it appears, without distinguishing loose and mounted (a simplification of this exploration). Only loose and mounted pieces enter, with the states written in the support-set study on V2 (Section 5.1). caos_f0211 is left out, because its body is held in the hand.

Criterion, written before running. A study is worth it if the mIoU over the 10 pieces improves at the table, at the pick and at the mounting, and arm_2 improves while arm_1 does not get worse. Otherwise the rule stays a recommendation for a new assembly.

The validation frames were already used to choose the current support set, so these values are optimistic (Cawley and Talbot 2010).

In [ ]:
# 1 Imports, paths and shared module
%load_ext autoreload
%autoreload 2
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(PROJECT_ROOT / "src"))
MODEL_DIR = PROJECT_ROOT / "models" / "c_radio_weights"
FRAMES_DIR = PROJECT_ROOT / "data" / "extracted_frames"
ANNOTATION_ARCHIVE = PROJECT_ROOT / "data" / "annotation_versions" / "margin_study"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "scratch" / "fine_details"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from perception.classes import CLASS_NAMES, HAND_LABEL, PIECE_LABELS
from perception.features import FeatureExtractor
from perception.labels import sequence_of, rasterize_annotation, patch_label_grid
from perception.prototypes import patch_prototypes, sort_by_class, class_similarities_gpu, assign_labels
from perception.metrics import confusion_matrix, frame_iou, frame_miou
from perception.visualization import render_overlay
from perception.prototypes import l2_normalize

In [2]:
# 2 Configuration
DEVICE = "cuda"
PATCH_SIZE = 16                      # patch size of C-RADIOv4
MIN_LABEL_COVERAGE = 0.5             # labelling rule of the pipeline (Section 5)
MARGIN = 0.0                         # background margin of V2 (margin study, Section 9)
GT_LABELS = PIECE_LABELS | {HAND_LABEL}
RESOLUTIONS = [1024, 1152, 2048]     # native; largest training size (tech report Sect. 2.2); 2x zoom as proxy for a second look
ROW_CHUNK = 2048                     # patches per GPU block for the similarities: bounds memory, does not change the result
TEST_SEQUENCES = {"assembly_1"}

# Frames chosen before looking; FOCUS = pieces the crop of Section 5 is centred on
FRAMES = {
    "assembly_2_f0005": ("table: loose mouth seen from the side", ["mouth"]),
    "assembly_2_f0057": ("table: both arms loose",                ["arm_1", "arm_2"]),
    "assembly_2_f0059": ("pick: arm_1",                           ["arm_1"]),
    "assembly_2_f0060": ("mount: arm_1",                          ["arm_1"]),
    "assembly_2_f0063": ("pick: arm_2",                           ["arm_2"]),
    "assembly_2_f0065": ("mount: arm_2",                          ["arm_2"]),
    "assembly_2_f0068": ("table: both arms mounted",              ["arm_1", "arm_2"]),
}

# Current support set: the 10 frames of the multi-prototype and margin studies, in the same order,
# with the annotations used there (phase_1_support_set_v2_study.ipynb, cell 2.5).
# caos_f0211 was completed in v1.7, so its old annotation is read from the archive.
CURRENT_SUPPORT = [
    {"frame": "5_assembled_f0002",            "use_labels": PIECE_LABELS},
    {"frame": "caos_f0149",                   "use_labels": PIECE_LABELS},
    {"frame": "5_assembled_f0019",            "use_labels": PIECE_LABELS},
    {"frame": "3_disassembled_ordered_f0001", "use_labels": PIECE_LABELS - {"base"}},
    {"frame": "2_disassembled_caos_f0016",    "use_labels": PIECE_LABELS - {"base"}},
    {"frame": "Take_3_1_Base_f0099",          "use_labels": {"base"}},
    {"frame": "Take_3_1_Base_f0131",          "use_labels": {"base"}},
    {"frame": "caos_f0211",                   "use_labels": {HAND_LABEL},
     "json_path": ANNOTATION_ARCHIVE / "caos_f0211.json"},
    {"frame": "4_partially_assembled_f0002",  "use_labels": {"body", "mouth"}},
    {"frame": "caos_f0043",                   "use_labels": {"mouth"}},
]
assert len(CURRENT_SUPPORT) == 10, "The support set must have the 10 frames of v1.7"

for item in CURRENT_SUPPORT:
    assert sequence_of(item["frame"]) not in TEST_SEQUENCES | {"assembly_2"}, f"Leakage: {item['frame']}"
    item["image_path"] = FRAMES_DIR / f"{item['frame']}.jpg"
    item["json_path"] = item.get("json_path") or FRAMES_DIR / f"{item['frame']}.json"
for frame in FRAMES:
    assert sequence_of(frame) not in TEST_SEQUENCES

extractor = FeatureExtractor(MODEL_DIR, device=DEVICE)

Loading weights:   0%|          | 0/330 [00:00<?, ?it/s]

In [3]:
# 3 Candidates of the support-set study on V2 with their states (Section 5.1), and the 32 validation frames (Section 2.5)
M, L = "mounted", "loose"
_LOOSE_NO_BASE = {piece: {L} for piece in PIECE_LABELS - {"base"}}      # disassembled frames: base split, not used
STATES = {
    "5_assembled_f0002":  {piece: {M} for piece in PIECE_LABELS},
    "5_assembled_f0019":  {piece: {M} for piece in PIECE_LABELS},
    "caos_f0278":         {piece: {M} for piece in PIECE_LABELS},
    "caos_f0149":         {"hat": {M}, "eyes": {M}, "mouth": {M}, "ear": {M, L},
                           "body": {L}, "base": {L}, "nose": {L}, "moustache": {L}, "arm_1": {L}},
    "caos_f0043":         {"body": {M}, "base": {M}, "eyes": {M}, "ear": {M, L},
                           "hat": {L}, "nose": {L}, "moustache": {L}, "mouth": {L}, "arm_1": {L}, "arm_2": {L}},
    "4_partially_assembled_f0002": {"body": {M}, "base": {M}, "eyes": {M}, "ear": {M, L}, "arm_1": {M},
                                    "mouth": {L}, "nose": {L}, "moustache": {L}, "hat": {L}, "arm_2": {L}},
    "3_disassembled_ordered_f0001":         _LOOSE_NO_BASE,
    "2_disassembled_caos_f0016":            _LOOSE_NO_BASE,
    "1_disassembled_controlled_caos_f0001": _LOOSE_NO_BASE,
    "1_disassembled_controlled_caos_f0016": _LOOSE_NO_BASE,
    "Take_3_1_Base_f0099": {"base": {L}},
    "Take_3_1_Base_f0131": {"base": {L}},
}   # caos_f0211 is left out: its body is held in the hand, not a loose or mounted state
DISASSEMBLED = {"3_disassembled_ordered", "2_disassembled_caos", "1_disassembled_controlled_caos"}
CANDIDATES = [{"frame": frame,
               "use_labels": PIECE_LABELS - ({"base"} if sequence_of(frame) in DISASSEMBLED else set()),
               "image_path": FRAMES_DIR / f"{frame}.jpg", "json_path": FRAMES_DIR / f"{frame}.json"}
              for frame in STATES]

_VALIDATION = [(3, "table"), (4, "pick"), (5, "mount"), (8, "table"), (10, "pick"), (12, "mount"), (17, "table"),
               (18, "pick"), (19, "mount"), (24, "table"), (26, "pick"), (28, "pick"), (30, "mount"), (33, "mount"),
               (37, "table"), (42, "pick"), (43, "mount"), (46, "table"), (50, "pick"), (51, "mount"), (52, "table"),
               (53, "pick"), (54, "mount"), (57, "table"), (59, "pick"), (60, "mount"), (63, "pick"), (65, "mount"),
               (68, "table"), (70, "pick"), (73, "mount"), (77, "table")]
VALIDATION_FRAMES = [{"frame": f"assembly_2_f{index:04d}", "moment": moment,
                      "image_path": FRAMES_DIR / f"assembly_2_f{index:04d}.jpg",
                      "json_path": FRAMES_DIR / f"assembly_2_f{index:04d}.json"} for index, moment in _VALIDATION]
EXCLUDED = [set() for _ in VALIDATION_FRAMES]

for item in CANDIDATES:
    assert sequence_of(item["frame"]) not in TEST_SEQUENCES | {"assembly_2"}, f"Leakage: {item['frame']}"
assert len(VALIDATION_FRAMES) == 32 and len(CANDIDATES) == 12
print(f"{len(CURRENT_SUPPORT)} reference frames, {len(CANDIDATES)} candidates, {len(VALIDATION_FRAMES)} validation frames")

10 reference frames, 12 candidates, 32 validation frames


In [4]:
# 4 Features at 1024; current V2; regression check against the support-set study on V2 (Observation 8.2)
def patch_labels(json_path, use_labels, size):
    """Patch labels of one annotation (rule of the pipeline, Section 5)."""
    return patch_label_grid(rasterize_annotation(json_path, use_labels, size), PATCH_SIZE, MIN_LABEL_COVERAGE)[0]


for item in CURRENT_SUPPORT + CANDIDATES:
    item["features"], size = extractor.extract(Image.open(item["image_path"]).convert("RGB"))
    item["patch_labels"] = patch_labels(item["json_path"], item["use_labels"], size)
for val in VALIDATION_FRAMES:
    val["features"], size = extractor.extract(Image.open(val["image_path"]).convert("RGB"))
    val["gt"] = patch_labels(val["json_path"], GT_LABELS, size)

CURRENT_CPU = patch_prototypes(CURRENT_SUPPORT)
assert len(CURRENT_CPU[1]) == 39125
MOMENT_FRAMES = {m: [k for k, val in enumerate(VALIDATION_FRAMES) if val["moment"] == m] for m in ("table", "pick", "mount")}
PIECES = [name for name in CLASS_NAMES if name in PIECE_LABELS]


def evaluate(prototypes, owners):
    """V2 at margin 0 on the 32 validation frames: mIoU over all frames, by moment, arms merged, and IoU of each piece."""
    v2 = tuple(torch.as_tensor(a, device=DEVICE) for a in sort_by_class(prototypes, owners))
    confusions = [confusion_matrix(val["gt"], assign_labels(class_similarities_gpu(val["features"], *v2), MARGIN))
                  for val in VALIDATION_FRAMES]
    iou = frame_iou(confusions, EXCLUDED)
    return {"all frames": frame_miou(confusions, EXCLUDED),
            **{m: frame_miou(confusions, EXCLUDED, frames) for m, frames in MOMENT_FRAMES.items()},
            "arms merged": frame_miou(confusions, EXCLUDED, merge=True),
            **{name: iou[CLASS_NAMES.index(name)] for name in PIECES}}


CURRENT_RESULT = evaluate(*CURRENT_CPU)
REPORTED = {"all frames": 0.779, "table": 0.818, "pick": 0.777, "mount": 0.731, "arms merged": 0.836}
regression = pd.DataFrame([{"metric": k, "obtained": round(CURRENT_RESULT[k], 3), "reported": v,
                            "equal": round(CURRENT_RESULT[k], 3) == v} for k, v in REPORTED.items()])
display(regression)
assert regression["equal"].all(), "The current support set must reproduce the support-set study on V2"

,metric,obtained,reported,equal
0,all frames,0.779,0.779,True
1,table,0.818,0.818,True
2,pick,0.777,0.777,True
3,mount,0.731,0.731,True
4,arms merged,0.836,0.836,True


In [5]:
# 5 Typical size of each piece in use (median over the validation frames where it appears), and the candidate
#   examples at least that large that are not yet in the support set
def size_of(labels, name):
    return int((labels == CLASS_NAMES.index(name)).sum())


USE_SIZE = {p: float(np.median([size_of(val["gt"], p) for val in VALIDATION_FRAMES if size_of(val["gt"], p) > 0]))
            for p in PIECES}
IN_SUPPORT = {(item["frame"], p) for item in CURRENT_SUPPORT for p in item["use_labels"] & PIECE_LABELS}

rows, ADDED = [], []
for item in CANDIDATES:
    for p in PIECES:
        n = size_of(item["patch_labels"], p)
        if p not in STATES[item["frame"]] or n == 0:
            continue
        large, used = n >= USE_SIZE[p], (item["frame"], p) in IN_SUPPORT
        rows.append({"piece": p, "frame": item["frame"], "state": " and ".join(sorted(STATES[item["frame"]][p])),
                     "patches": n, "size in use": USE_SIZE[p], "large enough": large, "already in support": used})
        if large and not used:
            ADDED.append((item, p))
examples = pd.DataFrame(rows).sort_values(["piece", "patches"], ascending=[True, False])
display(examples.set_index(["piece", "frame"]))

summary = pd.DataFrame([{"piece": p, "size in use": USE_SIZE[p],
                         "largest in support": max([size_of(item["patch_labels"], p) for item in CURRENT_SUPPORT], default=0),
                         "examples added": sum(1 for _, q in ADDED if q == p)} for p in PIECES]).set_index("piece")
display(summary)
print(f"{len(ADDED)} examples added, {sum(size_of(item['patch_labels'], p) for item, p in ADDED)} prototypes")

state  patches  size in use  \
piece frame                                                                 
arm_1 caos_f0149                              loose       16         10.0   
      caos_f0278                            mounted       10         10.0   
      4_partially_assembled_f0002           mounted        7         10.0   
      5_assembled_f0002                     mounted        5         10.0   
      5_assembled_f0019                     mounted        5         10.0   
...                                             ...      ...          ...   
nose  3_disassembled_ordered_f0001            loose        5          7.0   
      4_partially_assembled_f0002             loose        3          7.0   
      2_disassembled_caos_f0016               loose        3          7.0   
      1_disassembled_controlled_caos_f0016    loose        2          7.0   
      1_disassembled_controlled_caos_f0001    loose        1          7.0   

                                            large enough  already in support  
piece frame                                                                   
arm_1 caos_f0149                                    True                True  
      caos_f0278                                    True               False  
      4_partially_assembled_f0002                  False               False  
      5_assembled_f0002                            False                True  
      5_assembled_f0019                            False                True  
...                                                  ...                 ...  
nose  3_disassembled_ordered_f0001                 False                True  
      4_partially_assembled_f0002                  False               False  
      2_disassembled_caos_f0016                    False                True  
      1_disassembled_controlled_caos_f0016         False               False  
      1_disassembled_controlled_caos_f0001         False               False  

[97 rows x 5 columns]

,size in use,largest in support,examples added
piece,,,
body,36.0,109,4
base,23.0,59,2
hat,18.0,14,3
eyes,8.0,8,3
ear,8.0,13,3
nose,7.0,10,1
moustache,7.0,12,2
mouth,3.5,6,2
arm_1,10.0,16,1


23 examples added, 509 prototypes


In [6]:
# 6 Current support set plus the examples at working distance, against the current support set on the 32 frames
extra = [l2_normalize(item["features"])[item["patch_labels"] == CLASS_NAMES.index(p)] for item, p in ADDED]
extended = (np.concatenate([CURRENT_CPU[0]] + extra),
            np.concatenate([CURRENT_CPU[1]] + [np.full(len(v), CLASS_NAMES.index(p)) for v, (_, p) in zip(extra, ADDED)]))
EXTENDED_RESULT = evaluate(*extended)

comparison = pd.DataFrame({"current": CURRENT_RESULT, "with working distance": EXTENDED_RESULT})
comparison["difference"] = comparison["with working distance"] - comparison["current"]
display(comparison.round(3))

moments_better = all(EXTENDED_RESULT[m] > CURRENT_RESULT[m] for m in MOMENT_FRAMES)
arms_ok = EXTENDED_RESULT["arm_2"] > CURRENT_RESULT["arm_2"] and EXTENDED_RESULT["arm_1"] >= CURRENT_RESULT["arm_1"]
print(f"mIoU better at the table, the pick and the mounting: {moments_better}")
print(f"arm_2 better and arm_1 not worse: {arms_ok}")
print("Decision:", "a study of working distance is worth it" if moments_better and arms_ok
      else "the rule stays a recommendation for a new assembly")

,current,with working distance,difference
all frames,0.779,0.759,-0.020
table,0.818,0.775,-0.043
pick,0.777,0.756,-0.021
mount,0.731,0.749,0.018
arms merged,0.836,0.855,0.019
body,0.880,0.880,-0.001
base,0.906,0.922,0.016
hat,0.925,0.960,0.035
eyes,0.867,0.865,-0.002
ear,0.897,0.837,-0.059


mIoU better at the table, the pick and the mounting: False
arm_2 better and arm_1 not worse: False
Decision: the rule stays a recommendation for a new assembly


Observation 6: The regression check reproduces the support-set study on V2 (0.779; 0.818, 0.777 and 0.731 by moment; 0.836 with the arms merged). The rule adds 23 examples, 509 prototypes. Only two pieces had no example in the support set as large as in use: hat (largest 14 patches against 18) and arm_2 (4 against 6).

With the added examples the mIoU over 10 pieces falls from 0.779 to 0.759: from 0.818 to 0.775 at the table and from 0.777 to 0.756 at the pick, while it rises from 0.731 to 0.749 at the mounting. With the arms merged it rises from 0.836 to 0.855. Nose (+0.077), hat (+0.035), moustache (+0.033), base (+0.016) and arm_2 (+0.085) improve; ear (−0.059) and above all arm_1 (from 0.741 to 0.363) get worse. By the criterion written before running, the rule stays a recommendation for a new assembly.

Hypothesis: examples at the scale of use help most pieces (the mean of the eight pieces other than the arms goes from 0.834 to 0.846), but adding arm examples moves the confusion between the two arms again, as in the support-set study on V2 (Obs. 9.2) and in the fine-details exploration. The identity of two almost identical pieces is not solved by the examples; it is left to Phase 2.

In [ ]:
# 7 Support set without the two mouth frames (4_partially_assembled_f0002, caos_f0043): recognition and time
# Rule, written before running: remove them if the mIoU over 10 pieces and the IoU of the mouth do not fall
# (0.779 and 0.662, at three decimals). A tie keeps the smaller support set.
EIGHT = [item for item in CURRENT_SUPPORT if item["frame"] not in {"4_partially_assembled_f0002", "caos_f0043"}]
assert len(EIGHT) == 8
EIGHT_CPU = patch_prototypes(EIGHT)
EIGHT_RESULT = evaluate(*EIGHT_CPU)

def classification_ms(prototypes, owners, repeats=20):
    """Median time of V2 classification on one validation frame, GPU, after warm-up."""
    v2 = tuple(torch.as_tensor(a, device=DEVICE) for a in sort_by_class(prototypes, owners))
    features = VALIDATION_FRAMES[0]["features"]
    for _ in range(3):
        class_similarities_gpu(features, *v2)
    times = []
    for _ in range(repeats):
        torch.cuda.synchronize(); start = time.perf_counter()
        assign_labels(class_similarities_gpu(features, *v2), MARGIN)
        torch.cuda.synchronize(); times.append((time.perf_counter() - start) * 1000)
    return float(np.median(times))

table = pd.DataFrame({"ten frames": CURRENT_RESULT, "eight frames": EIGHT_RESULT})
table["difference"] = table["eight frames"] - table["ten frames"]
display(table.round(3))
print(f"Prototypes: {len(CURRENT_CPU[1])} with ten frames, {len(EIGHT_CPU[1])} with eight")
print(f"Classification time (median): {classification_ms(*CURRENT_CPU):.1f} ms with ten, {classification_ms(*EIGHT_CPU):.1f} ms with eight")
remove = (round(EIGHT_RESULT['all frames'], 3) >= round(CURRENT_RESULT['all frames'], 3)
          and round(EIGHT_RESULT['mouth'], 3) >= round(CURRENT_RESULT['mouth'], 3))
print("Decision:", "remove the two mouth frames" if remove else "keep the ten frames")

Observation 7: Without 4_partially_assembled_f0002 and caos_f0043 the mIoU over 10 pieces falls from 0.779 to 0.754, at the table (−0.022), at the pick (−0.025) and at the mounting (−0.030). The mouth falls from 0.662 to 0.514, nose by 0.043, body by 0.023, base and moustache by 0.019; the arms and the hat barely change. The prototypes drop from 39,125 to 31,904 and classification from 76 to 66 ms. By the rule written before running, the ten frames are kept.

The two frames already improved one prototype per class in the multi-prototype study (Obs. 5.2). The test does not separate the effect of their mouth patches from that of their background and body patches.

-------------------------------------------------------------------------------------------------------------------------------------------